# Assignment 3 — Statistical Analysis
### Retail Sales Dataset  |  100 Marks  |  7 Tasks

---

**Prerequisite:** Run `A1_Data_Cleaning.ipynb` first.

| Assignment | Notebook |
|---|---|
| Assignment 1 | Data Cleaning |
| Assignment 2 | Exploratory Data Analysis |
| Assignment 3 | Statistical Analysis |
| Assignment 4 | Segmentation and Clustering |
| Assignment 5 | Time Series Forecasting |


## Environment Setup

In [ ]:
# Run once to install any missing packagesimport subprocess, syspkgs = ['pandas','numpy','matplotlib','seaborn','scipy','statsmodels','scikit-learn','pmdarima','prophet']for p in pkgs:subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)print("All packages ready.")

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# Plot stylesns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(12,5),'axes.titlesize':13, 'axes.labelsize':11})print("Imports complete ")

# Assignment 3 — Statistical Analysis
**Marks: 100  |  Tasks: 7**

---

## Why Formal Statistics?

EDA reveals patterns visually — but visualization is subjective. Two analysts looking at the same box plot may disagree on whether the medians are "really different." Formal statistical testing provides an objective, reproducible decision rule: given this data, how likely is it that the observed difference occurred by chance?

Statistical analysis transforms business questions into testable hypotheses. Instead of "it looks like the South region has higher returns," you can state: "The South region's return rate (18.3%) is statistically significantly higher than other regions' return rate (11.2%) (Chi-square test, p = 0.003)."

### The Hypothesis Testing Framework

Every statistical test follows the same logical structure:

1. **State the null hypothesis (H0):** The default assumption — usually "no effect," "no difference," or "no relationship"
2. **State the alternative hypothesis (H1):** The claim you want to test
3. **Choose a significance level (alpha):** Typically 0.05 — the probability of a Type I error you are willing to accept
4. **Compute the test statistic and p-value**
5. **Make a decision:** If p < alpha, reject H0; otherwise fail to reject H0

### What the P-value Actually Means

The p-value is the probability of observing results at least as extreme as your data *assuming H0 is true*. It is NOT:
- The probability that H0 is true
- The probability that you made an error
- A measure of the size or importance of the effect

A small p-value means the data is unlikely under H0. Combined with a large effect size, it means the finding is both statistically significant AND practically meaningful.

### Task Overview

| Task | Test / Method | Question Answered |
|---|---|---|
| 1 | Shapiro-Wilk, D'Agostino, Q-Q plot | Is the data normally distributed? |
| 2 | t-confidence intervals, bootstrap CI | How precisely can we estimate the mean? |
| 3 | t-test, Mann-Whitney U | Do two groups have different distributions? |
| 4 | ANOVA, Tukey HSD | Do three or more groups differ? |
| 5 | Pearson, Spearman, partial correlation | How strongly are two variables associated? |
| 6 | OLS regression | What predicts revenue? By how much? |
| 7 | Chi-square, Cramer's V | Are two categorical variables associated? |


In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import stats as spimport statsmodels.api as smimport statsmodels.formula.api as smffrom statsmodels.stats.multicomp import pairwise_tukeyhsdfrom statsmodels.stats.outliers_influence import variance_inflation_factorimport warningswarnings.filterwarnings('ignore')sns.set_theme(style='whitegrid', font_scale=1.05)plt.rcParams.update({'figure.dpi':110})df = pd.read_csv('retail_sales_clean.csv', parse_dates=['order_date'])for col in ['product_category','region','gender','payment_method','order_status']:df[col] = df[col].astype('category')print(f"Loaded {df.shape[0]:,} rows. Ready for statistical analysis.")

---
## Task 1 — Distribution Testing (Normality)

### Theory: Why Normality Matters

Many parametric statistical tests (t-test, ANOVA, Pearson correlation, OLS regression) assume that the data — or the residuals — follow a normal distribution. Violating this assumption does not always invalidate the test (thanks to the Central Limit Theorem for large samples), but knowing whether your data is normal guides:
- Which test to use (parametric vs non-parametric)
- Whether transformations are needed
- How to interpret confidence intervals

### The Normal Distribution

The normal distribution (Gaussian) is defined by two parameters: mean (mu) and standard deviation (sigma). It is:
- Perfectly symmetric around the mean
- Bell-shaped
- Characterized by the 68-95-99.7 rule: 68% of data within 1 sigma, 95% within 2 sigma, 99.7% within 3 sigma

In practice, very few real-world variables are perfectly normal. The question is whether the departure from normality is *material* for the test you want to apply.

### Normality Tests

| Test | Best For | Null Hypothesis | Notes |
|---|---|---|---|
| **Shapiro-Wilk** | n < 2000 | Data is normal | Most powerful test for small-to-medium samples |
| **D'Agostino-Pearson (K-squared)** | n > 50 | Data is normal | Tests both skewness and kurtosis simultaneously |
| **Kolmogorov-Smirnov** | Any n | Data follows specified distribution | Less powerful than Shapiro-Wilk for normality |
| **Q-Q Plot** | Visual check | N/A | Always pair with a formal test |

**Caution for large samples:** With n > 2000, even tiny deviations from normality produce p < 0.05. A Shapiro-Wilk rejection on n = 5000 often means the data is *approximately* normal — which is fine for t-tests and ANOVA. Use the Q-Q plot to assess the *magnitude* of the departure.

### Reading a Q-Q Plot

A Q-Q (Quantile-Quantile) plot compares the sorted sample quantiles against the theoretical quantiles of a normal distribution. If the data is perfectly normal, all points fall exactly on the diagonal reference line.

Common patterns and what they mean:
- **S-shaped curve:** The distribution is skewed
- **Points bowing above the line at both ends:** Heavy tails (kurtosis > 0)
- **Points bowing below the line at both ends:** Light tails (kurtosis < 0)
- **Flat section followed by steep:** Bimodal distribution

### When Normality is Violated

If the data is significantly non-normal:
1. **Try log transformation:** `np.log1p(x)` — fixes right-skewed distributions
2. **Try square root transformation:** `np.sqrt(x)` — milder fix for moderate skewness
3. **Use non-parametric tests:** Mann-Whitney U instead of t-test; Kruskal-Wallis instead of ANOVA
4. **Rely on CLT:** For large samples (n > 100 per group), t-tests and ANOVA are robust to non-normality


In [ ]:
def normality_report(series, name, sample_n=500):"""Run three normality tests and print a structured report."""# Use a random sample for Shapiro-Wilk (designed for small-medium n)s = series.dropna()sample = s.sample(min(sample_n, len(s)), random_state=42)sw_stat, sw_p = sp.shapiro(sample)dp_stat, dp_p = sp.normaltest(sample)print(f"\n{'='*50}")print(f" Normality Test: {name}")print(f" n = {len(s):,} | sample for Shapiro = {len(sample)}")print(f"{'='*50}")print(f" Shapiro-Wilk : W={sw_stat:.4f} p={sw_p:.6f} "f"{'→ NOT Normal ' if sw_p<0.05 else '→ Normal '}")print(f" D'Agostino-P. : stat={dp_stat:.4f} p={dp_p:.6f} "f"{'→ NOT Normal ' if dp_p<0.05 else '→ Normal '}")print(f" Skewness: {s.skew():.3f} Kurtosis: {s.kurt():.3f}")for col in ['sales_amount','profit','age']:normality_report(df[col], col)

In [ ]:
# Q-Q plots for 3 columnsfig, axes = plt.subplots(2, 3, figsize=(15, 8))for i, col in enumerate(['sales_amount','profit','age']):# Originalsp.probplot(df[col].dropna(), dist='norm', plot=axes[0, i])axes[0, i].set_title(f'Q-Q Plot: {col}\n(original)')# Log-transformedlog_data = np.log1p(df[col].dropna().clip(lower=0))sp.probplot(log_data, dist='norm', plot=axes[1, i])axes[1, i].set_title(f'Q-Q Plot: log(1+{col})')plt.suptitle('Q-Q Plots — Points on diagonal line = Normal distribution', fontsize=12, y=1.01)plt.tight_layout()plt.show()print("Interpretation: If points deviate from the red line (especially at tails),")print("the distribution is non-normal. Log-transformation often improves this.")

In [ ]:
# Log transformation of sales_amountdf['log_sales'] = np.log1p(df['sales_amount'])fig, axes = plt.subplots(1, 2, figsize=(13, 4))sns.histplot(df['sales_amount'], kde=True, bins=50, ax=axes[0], color='steelblue')axes[0].set_title(f'sales_amount (skew={df["sales_amount"].skew():.2f})')sns.histplot(df['log_sales'], kde=True, bins=50, ax=axes[1], color='darkorange')axes[1].set_title(f'log(1+sales_amount) (skew={df["log_sales"].skew():.2f})')plt.suptitle('Log Transformation Effect on sales_amount', fontsize=12, y=1.01)plt.tight_layout()plt.show()print("After log-transform, skewness reduced from",f"{df['sales_amount'].skew():.2f} to {df['log_sales'].skew():.2f}")

---
## Task 2 — Confidence Intervals

### Theory: Quantifying Estimation Uncertainty

A sample mean is a single number (a point estimate). But if you drew a different sample, you would get a different mean. The confidence interval (CI) quantifies this sampling uncertainty — it gives a range of plausible values for the true population mean.

### What a 95% Confidence Interval Means

A 95% CI means: if we repeated the sampling and CI-construction process 100 times, approximately 95 of the resulting intervals would contain the true population mean.

It does NOT mean: "there is a 95% probability that the true mean lies within this particular interval." Once the interval is computed, the true mean either is or is not inside it — probability is not the right framing.

### The Formula

For large samples (n > 30) or normal data:

```
CI = x_bar +/- t*(alpha/2, n-1) * (s / sqrt(n))
```

Where:
- `x_bar` = sample mean
- `t*(alpha/2, n-1)` = t-distribution critical value (approximately 1.96 for 95% CI with large n)
- `s` = sample standard deviation
- `n` = sample size
- `s / sqrt(n)` = standard error of the mean (SE)

### The Standard Error

The standard error (SE = s / sqrt(n)) measures how much the sample mean would vary across repeated samples. As n increases, SE shrinks — larger samples give more precise estimates. Doubling n reduces SE by a factor of sqrt(2) ≈ 1.41 — to halve the SE, you need to quadruple the sample size.

### Interpreting CIs in Practice

| CI Width | Meaning |
|---|---|
| Narrow CI | High precision — the sample mean is a reliable estimate of the population mean |
| Wide CI | Low precision — collect more data |
| CI that excludes 0 (for differences) | Significant difference at the corresponding alpha level |
| Overlapping CIs for two groups | Cannot conclude the groups differ (but also not proof they are the same) |

### Bootstrap Confidence Intervals

For non-normal data or statistics without closed-form CIs (e.g., median, 90th percentile), bootstrapping is the universal solution:

1. Resample n observations with replacement from the data (B = 5000 times)
2. Compute the statistic of interest for each resample
3. The 2.5th and 97.5th percentiles of the resulting bootstrap distribution are the 95% CI

Bootstrapped CIs make no normality assumptions and work for any statistic.


In [ ]:
# Overall 95% CI for mean sales_amountdef mean_ci_95(series):s = series.dropna()lo, hi = sp.t.interval(0.95, df=len(s)-1,loc=s.mean(), scale=sp.sem(s))return s.mean(), lo, hioverall_mean, lo, hi = mean_ci_95(df['sales_amount'])print(f"Overall mean sales_amount: ₹{overall_mean:,.2f}")print(f"95% Confidence Interval: [₹{lo:,.2f} to ₹{hi:,.2f}]")print(f"CI width: ₹{hi-lo:,.2f}")print("\nInterpretation: We are 95% confident the true mean sales_amount")print(f"in the population lies between ₹{lo:,.0f} and ₹{hi:,.0f}.")

In [ ]:
# 95% CI per region — error bar chartregion_ci = {}for region in df['region'].cat.categories:sales = df.loc[df['region']==region, 'sales_amount']mean, lo, hi = mean_ci_95(sales)region_ci[region] = {'mean': mean, 'lo': lo, 'hi': hi,'err_lo': mean-lo, 'err_hi': hi-mean}ci_df = pd.DataFrame(region_ci).T.sort_values('mean', ascending=False)fig, ax = plt.subplots(figsize=(10, 5))ax.errorbar(ci_df.index, ci_df['mean'],yerr=[ci_df['err_lo'], ci_df['err_hi']],fmt='o', capsize=8, capthick=2, elinewidth=2,markersize=10, color='steelblue', ecolor='#e74c3c')for i, (region, row) in enumerate(ci_df.iterrows()):ax.annotate(f"₹{row['mean']:,.0f}", (i, row['mean']),textcoords='offset points', xytext=(12,0), fontsize=9)ax.set_title('Mean Sales Amount by Region\n(Error bars = 95% Confidence Interval)')ax.set_ylabel('Mean Sales Amount (INR)')ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))plt.tight_layout()plt.show()print("\nDo any CIs NOT overlap? Non-overlapping CIs → likely statistically significant difference.")print("Overlapping CIs → cannot conclude a significant difference without a formal test.")

---
## Task 3 — Two-Sample Hypothesis Tests

### Theory: Choosing the Right Test

When comparing two groups, the choice of test depends on the variable type, distribution shape, and whether the two groups are independent or paired:

| Test | Variable Type | Normality | Groups | Null Hypothesis |
|---|---|---|---|---|
| **Welch's t-test** | Continuous | Required (or large n) | Independent | mu_1 = mu_2 (means equal) |
| **Paired t-test** | Continuous | Required (or large n) | Paired/matched | Mean difference = 0 |
| **Mann-Whitney U** | Continuous / Ordinal | Not required | Independent | Same distribution |
| **Wilcoxon signed-rank** | Continuous | Not required | Paired | Median difference = 0 |

### Welch's vs Student's t-test

**Student's t-test** assumes equal variances between the two groups. **Welch's t-test** does not — it adjusts the degrees of freedom using the Welch-Satterthwaite equation to account for unequal variances.

Always use Welch's t-test (`equal_var=False` in scipy) as the default. It performs nearly identically to Student's t when variances are equal, but is far more reliable when they are not. The equal-variance assumption is frequently violated in real data and is rarely worth the trouble of testing.

### The Mann-Whitney U Test

The Mann-Whitney U test is the non-parametric equivalent of the independent t-test. It ranks all observations regardless of group, then tests whether the average rank differs between groups. A significant result means the two groups have different distributions — typically interpreted as different medians.

Use Mann-Whitney when:
- The data is not normally distributed (small samples where CLT does not apply)
- The variable is ordinal (e.g., satisfaction ratings on a 1-5 scale)
- Outliers are present that you do not want to remove or Winsorize

### Effect Size — Cohen's d

A p-value tells you whether an effect is statistically significant. Cohen's d tells you whether it is practically meaningful:

```
d = (mean_1 - mean_2) / pooled_std
```

| d value | Interpretation |
|---|---|
| 0.2 | Small effect — often not worth acting on |
| 0.5 | Medium effect |
| 0.8 | Large effect |
| > 1.0 | Very large effect |

A result can be statistically significant (small p-value) with a trivial effect size (d = 0.05) when the sample size is very large. Always report both.

### Type I and Type II Errors

| Error Type | Definition | Consequence | Controlled By |
|---|---|---|---|
| Type I (false positive) | Reject H0 when it is true | Act on a non-existent effect | Significance level alpha |
| Type II (false negative) | Fail to reject H0 when it is false | Miss a real effect | Statistical power (1 - beta) |

Most studies aim for alpha = 0.05 (5% Type I error rate) and power >= 0.80 (20% Type II error rate).


In [ ]:
def two_sample_report(group1, group2, label1, label2, var_name):"""Full two-sample test report with automatic test selection."""print(f"\n{'='*60}")print(f" {var_name}: {label1} vs {label2}")print(f" n₁={len(group1):,} mean₁=₹{group1.mean():,.0f}")print(f" n₂={len(group2):,} mean₂=₹{group2.mean():,.0f}")print(f"{'='*60}")# Normality_, sw_p1 = sp.shapiro(group1.sample(min(200, len(group1)), random_state=1))_, sw_p2 = sp.shapiro(group2.sample(min(200, len(group2)), random_state=1))normal = (sw_p1 > 0.05) and (sw_p2 > 0.05)# Levene_, lev_p = sp.levene(group1, group2)equal_var = lev_p > 0.05print(f" Shapiro-Wilk: {label1} p={sw_p1:.4f}, {label2} p={sw_p2:.4f}")print(f" Levene's (equal variance): p={lev_p:.4f} → {'Equal' if equal_var else 'Unequal'} variance")# Choose testif normal:stat, p = sp.ttest_ind(group1, group2, equal_var=equal_var)test_name = "Welch's t-test" if not equal_var else "Independent t-test"else:stat, p = sp.mannwhitneyu(group1, group2, alternative='two-sided')test_name = "Mann-Whitney U"# Effect size (Cohen's d)pooled_std = np.sqrt((group1.std()**2 + group2.std()**2) / 2)cohens_d = (group1.mean() - group2.mean()) / pooled_stdeffect = 'small' if abs(cohens_d)<0.5 else ('medium' if abs(cohens_d)<0.8 else 'large')print(f" Test chosen: {test_name}")print(f" H₀: mean({label1}) = mean({label2})")print(f" H₁: mean({label1}) ≠ mean({label2})")print(f" Statistic={stat:.4f} p-value={p:.6f}")print(f" Decision: {'REJECT H₀ ' if p<0.05 else 'FAIL TO REJECT H₀'} (α=0.05)")print(f" Cohen's d = {cohens_d:.3f} → {effect} effect")# Test A: Male vs Female sales_amountg1 = df.loc[df['gender']=='Male', 'sales_amount']g2 = df.loc[df['gender']=='Female','sales_amount']two_sample_report(g1, g2, 'Male','Female','sales_amount')

In [ ]:
# Test B: returned vs not-returned customer_satisfaction (Mann-Whitney: ordinal)g1 = df.loc[df['return_flag']==True, 'customer_satisfaction'].dropna()g2 = df.loc[df['return_flag']==False, 'customer_satisfaction'].dropna()stat, p = sp.mannwhitneyu(g1, g2, alternative='two-sided')print(f"\ncustomer_satisfaction: Returned vs Not Returned")print(f"H₀: satisfaction is the same for returned and non-returned orders")print(f"Mann-Whitney U = {stat:.2f} p = {p:.6f}")print(f"Decision: {'REJECT H₀ — significant difference' if p<0.05 else 'No significant difference'}")print(f"Mean satisfaction — Returned: {g1.mean():.2f} Not Returned: {g2.mean():.2f}")# Test C: Electronics vs Clothing profitg1 = df.loc[df['product_category']=='Electronics','profit']g2 = df.loc[df['product_category']=='Clothing', 'profit']stat, p = sp.ttest_ind(g1, g2, equal_var=False)print(f"\nprofit: Electronics vs Clothing (Welch's t-test)")print(f"H₀: mean profit is equal for Electronics and Clothing")print(f"t = {stat:.4f} p = {p:.6f}")print(f"Decision: {'REJECT H₀ ' if p<0.05 else 'Fail to Reject H₀'}")print(f"Mean profit — Electronics: ₹{g1.mean():,.0f} Clothing: ₹{g2.mean():,.0f}")

---
## Task 4 — ANOVA (Analysis of Variance)

### Theory: Testing Means Across Three or More Groups

When you have three or more groups, running multiple t-tests for every pair inflates the Type I error rate. With 5 groups, there are 10 pairwise comparisons. Running 10 tests at alpha = 0.05 means approximately a 40% chance of finding at least one false positive, even if all groups are truly identical.

One-Way ANOVA solves this by testing all groups simultaneously in a single test.

### How ANOVA Works

ANOVA partitions total variance into two sources:

| Source | Symbol | Meaning |
|---|---|---|
| Between-group variance | SSB | How much do group means differ from the grand mean? |
| Within-group variance | SSW | How much do individual values vary within each group? |

The F-statistic is the ratio of between-group to within-group variance:

```
F = MSB / MSW = [SSB / (k-1)] / [SSW / (N-k)]
```

Where k = number of groups, N = total observations.

If the group means are truly equal (H0 is true), MSB ≈ MSW and F ≈ 1. If at least one group mean differs, MSB > MSW and F > 1.

**H0:** All group means are equal (mu_1 = mu_2 = ... = mu_k)
**H1:** At least one group mean is different

### ANOVA Assumptions

1. **Independence:** Observations are independent within and across groups
2. **Normality:** Each group's values are approximately normally distributed
3. **Homogeneity of variance (homoscedasticity):** All groups have approximately equal variance (test with Levene's test)

ANOVA is robust to mild violations of normality (especially with balanced group sizes and n > 30 per group), but sensitive to severe heteroscedasticity. If Levene's test is significant, use Welch's ANOVA.

### Post-Hoc Tests — Tukey HSD

A significant ANOVA F-test tells you that *at least one* pair of groups differs — it does not tell you *which* pairs. Post-hoc tests perform all pairwise comparisons while controlling the family-wise error rate.

**Tukey's Honest Significant Difference (HSD)** is the most common post-hoc test. It adjusts the critical value using the Studentized Range Distribution, ensuring the overall Type I error rate across all pairwise comparisons stays at alpha.

### Two-Way ANOVA

When two categorical factors potentially influence the outcome, Two-Way ANOVA tests:
1. Main effect of Factor A (e.g., does region affect sales?)
2. Main effect of Factor B (e.g., does product category affect sales?)
3. Interaction effect A x B (e.g., does the effect of region depend on which product category?)

The interaction term is crucial: if it is significant, you cannot interpret the main effects in isolation.


In [ ]:
# Check ANOVA assumptions firstprint("=== ANOVA Assumption Check: sales_amount across regions ===")print("\n1. Normality (Shapiro-Wilk per group):")region_groups = [df.loc[df['region']==r, 'sales_amount'].sample(min(200,sum(df['region']==r)),random_state=42) for r in df['region'].cat.categories]for region, grp in zip(df['region'].cat.categories, region_groups):_, p = sp.shapiro(grp)print(f" {region}: p={p:.4f} {'Normal ' if p>0.05 else 'NOT Normal '}")print("\n2. Homogeneity of Variance (Levene's test):")lev_stat, lev_p = sp.levene(*region_groups)print(f" Levene's: F={lev_stat:.4f} p={lev_p:.4f} "f"{'Equal variances ' if lev_p>0.05 else 'Unequal variances '}")print(" Note: ANOVA is robust to mild violations with large equal samples.")

In [ ]:
# One-Way ANOVA: sales_amount across 5 regionsgroups_by_region = [df.loc[df['region']==r,'sales_amount'] for r in df['region'].cat.categories]f_stat, p_val = sp.f_oneway(*groups_by_region)print(f"One-Way ANOVA: sales_amount ~ region")print(f"H₀: μ_North = μ_South = μ_East = μ_West = μ_Central")print(f"H₁: At least one region mean differs")print(f"F = {f_stat:.4f} p = {p_val:.6f}")print(f"Decision: {'REJECT H₀ — at least one region mean significantly differs' if p_val<0.05 else 'Fail to Reject H₀'}")# Eta-squared (effect size)grand_mean = df['sales_amount'].mean()ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups_by_region)ss_total = sum((df['sales_amount'] - grand_mean)**2)eta_sq = ss_between / ss_totalprint(f"\nEta-squared (η²) = {eta_sq:.4f}")print(f"Interpretation: {eta_sq*100:.1f}% of total variance in sales_amount is explained by region.")

In [ ]:
# Tukey HSD post-hoc testtukey = pairwise_tukeyhsd(df['sales_amount'], df['region'], alpha=0.05)print(tukey.summary())# Visualise Tukey resultsfig, ax = plt.subplots(figsize=(8, 6))tukey.plot_simultaneous(ax=ax, comparison_name=df['region'].cat.categories[0])ax.set_title('Tukey HSD 95% Confidence Intervals\nNon-overlapping intervals = significant difference')plt.tight_layout()plt.show()

In [ ]:
# Two-Way ANOVA: sales_amount ~ region * product_categorymodel = smf.ols('sales_amount ~ C(region) + C(product_category) + C(region):C(product_category)',data=df).fit()anova_table = sm.stats.anova_lm(model, typ=2)print("Two-Way ANOVA Table:")print(anova_table.round(4).to_string())print("\nInterpretation:")for idx, row in anova_table.iterrows():print(f" {idx[:40]:<40} p = {row['PR(>F)']:.4f} "f"{'Significant ' if row['PR(>F)']<0.05 else 'Not significant'}")

---
## Task 5 — Correlation Analysis

### Theory: Three Measures of Association

Correlation measures the strength and direction of the statistical relationship between two variables. The three main correlation coefficients serve different purposes:

### Pearson Correlation (r)

Measures the strength of the **linear** relationship between two continuous variables.

```
r = sum((xi - x_bar)(yi - y_bar)) / sqrt(sum((xi - x_bar)^2) * sum((yi - y_bar)^2))
```

Properties:
- Range: -1 to +1
- r = +1: perfect positive linear relationship
- r = -1: perfect negative linear relationship
- r = 0: no linear relationship (but could have a strong non-linear one!)
- Assumes both variables are approximately normally distributed
- Sensitive to outliers (a single extreme point can dramatically change r)

### Spearman Rank Correlation (rho)

Measures the strength of the **monotonic** (not necessarily linear) relationship. It works by replacing the raw values with their ranks and then computing Pearson r on the ranks.

Properties:
- Same range (-1 to +1) and interpretation direction as Pearson r
- No normality assumption — works for any distribution
- Robust to outliers (outliers have bounded rank impact)
- Appropriate for ordinal variables

**When Pearson and Spearman disagree substantially:** The relationship is monotone but non-linear, or outliers are distorting the Pearson result.

### Partial Correlation

Partial correlation measures the association between two variables after removing (partialling out) the influence of one or more control variables. This is essential when a confounding variable creates a spurious correlation.

Example: age and revenue may be positively correlated in raw data. But if older customers buy more expensive products (higher unit price), the correlation between age and revenue might disappear after controlling for unit price. The partial correlation reveals whether age truly predicts revenue beyond the price effect.

### Correlation is Not Causation

This cannot be overstated. Ice cream sales and drowning deaths are positively correlated — because both increase in summer (confounded by temperature). Correlation analysis identifies association; establishing causation requires:
- Controlled experiments (A/B tests)
- Natural experiments (difference-in-differences)
- Structural causal models with domain knowledge

### Practical Thresholds

| |r| | Interpretation |
|---|---|
| 0.0 – 0.2 | Negligible or no relationship |
| 0.2 – 0.4 | Weak relationship |
| 0.4 – 0.6 | Moderate relationship |
| 0.6 – 0.8 | Strong relationship |
| 0.8 – 1.0 | Very strong relationship |


In [ ]:
# Pearson: sales_amount vs profitr, p = sp.pearsonr(df['sales_amount'], df['profit'])print(f"Pearson r (sales_amount vs profit): {r:.4f} p = {p:.2e}")effect = 'small' if abs(r)<0.1 else ('medium' if abs(r)<0.3 else ('large' if abs(r)<0.5 else 'very large'))print(f"Effect size: {effect} | {r*100:.1f}% of variance shared between the two variables (r²)")# Spearman: customer_satisfaction vs discount_pctrho, p = sp.spearmanr(df['customer_satisfaction'].dropna(),df.loc[df['customer_satisfaction'].notna(),'discount_pct'])print(f"\nSpearman ρ (satisfaction vs discount_pct): {rho:.4f} p = {p:.4f}")print("Why Spearman? satisfaction is ordinal (1-5 scale) — rank correlation is more appropriate.")print(f"Direction: {'negative' if rho<0 else 'positive'} — {'higher discount → lower satisfaction' if rho<0 else 'higher discount → higher satisfaction'}")# Point-Biserial: return_flag vs sales_amountpb_r, p = sp.pointbiserialr(df['return_flag'].astype(int), df['sales_amount'])print(f"\nPoint-Biserial r (return_flag vs sales_amount): {pb_r:.4f} p = {p:.4f}")print(f"Direction: {'Returned orders have HIGHER sales amounts' if pb_r>0 else 'Returned orders have LOWER sales amounts'}")

---
## Task 6 — Linear Regression

### Theory: Modeling Relationships with OLS

Linear regression models the relationship between one or more predictor variables (X) and a continuous outcome variable (Y) by finding the line (or hyperplane in multiple dimensions) that minimizes the sum of squared residuals (OLS — Ordinary Least Squares).

### Simple Linear Regression

```
Y_hat = beta_0 + beta_1 * X
```

- `beta_0` (intercept): The predicted value of Y when X = 0
- `beta_1` (slope): The change in Y for a one-unit increase in X

OLS minimizes sum((yi - y_hat_i)^2). The solution has a closed form:

```
beta_1 = sum((xi - x_bar)(yi - y_bar)) / sum((xi - x_bar)^2)
beta_0 = y_bar - beta_1 * x_bar
```

### Multiple Linear Regression

```
Y_hat = beta_0 + beta_1*X1 + beta_2*X2 + ... + beta_p*Xp
```

Each coefficient beta_i represents the change in Y per unit change in Xi, **holding all other predictors constant**. This "holding constant" interpretation is what gives regression its causal flavor — though correlation in observational data does not establish causation.

### Model Evaluation

| Metric | Formula | Interpretation |
|---|---|---|
| R-squared | 1 - SSR/SST | % of Y variance explained by the model |
| Adjusted R-squared | 1 - (1-R2)(n-1)/(n-p-1) | R-squared penalized for number of predictors |
| RMSE | sqrt(mean(residuals^2)) | Average prediction error (same units as Y) |
| MAE | mean(|residuals|) | Average absolute prediction error |
| F-statistic | Overall model significance | Does the model explain more than chance? |

R-squared always increases when you add predictors, even useless ones. Adjusted R-squared only increases if the new predictor genuinely improves the model.

### Regression Assumptions (LINE)

| Letter | Assumption | How to Check |
|---|---|---|
| L | **Linearity:** Y is a linear function of X | Scatter plot of Y vs X; residuals vs fitted |
| I | **Independence:** Residuals are uncorrelated | Durbin-Watson test; residuals vs time |
| N | **Normality:** Residuals are normally distributed | Q-Q plot of residuals; Shapiro-Wilk on residuals |
| E | **Equal variance (homoscedasticity):** Constant residual variance | Scale-location plot; Breusch-Pagan test |

Additionally, for multiple regression:
- **No multicollinearity:** Predictors are not highly correlated with each other (check VIF)

### VIF (Variance Inflation Factor)

VIF measures how much the variance of a regression coefficient is inflated due to multicollinearity:

```
VIF_j = 1 / (1 - R^2_j)
```

Where R^2_j is from regressing predictor j on all other predictors.

- VIF = 1: No multicollinearity
- VIF 1-5: Acceptable
- VIF 5-10: Moderate concern; investigate
- VIF > 10: Severe multicollinearity; the coefficient for this predictor is unreliable


In [ ]:
# Simple OLS: sales_amount ~ unit_pricemodel1 = smf.ols('sales_amount ~ unit_price', data=df).fit()print(model1.summary())

In [ ]:
# Multiple OLS: add quantity, discount_pct, days_to_shipmodel2 = smf.ols('sales_amount ~ unit_price + quantity + discount_pct + days_to_ship',data=df).fit()print(f"R² = {model2.rsquared:.4f} Adj R² = {model2.rsquared_adj:.4f}")print(f"F-statistic = {model2.fvalue:.4f} p = {model2.f_pvalue:.4e}")print("\nCoefficients:")print(model2.params.round(4).to_string())# VIF checkX = df[['unit_price','quantity','discount_pct','days_to_ship']].dropna()vif_data = pd.DataFrame({'Feature': X.columns,'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})print("\nVariance Inflation Factors (VIF > 10 = multicollinearity problem):")print(vif_data.to_string(index=False))

In [ ]:
# Regression assumption diagnosticsfitted = model2.fittedvaluesresid = model2.residstd_res = (resid - resid.mean()) / resid.std()fig, axes = plt.subplots(2, 2, figsize=(13, 9))# 1. Residuals vs Fittedaxes[0,0].scatter(fitted, resid, alpha=0.3, s=12, color='steelblue')axes[0,0].axhline(0, color='red', linewidth=1.5)axes[0,0].set_xlabel('Fitted Values'); axes[0,0].set_ylabel('Residuals')axes[0,0].set_title('Residuals vs Fitted\n(no pattern = linearity holds)')# 2. Q-Q plot of residualssp.probplot(resid, dist='norm', plot=axes[0,1])axes[0,1].set_title('Normal Q-Q of Residuals\n(diagonal = normal residuals)')# 3. Scale-Location (√|std_residuals| vs fitted)axes[1,0].scatter(fitted, np.sqrt(np.abs(std_res)), alpha=0.3, s=12, color='green')axes[1,0].set_xlabel('Fitted Values'); axes[1,0].set_ylabel('√|Std Residuals|')axes[1,0].set_title('Scale-Location\n(flat line = homoscedasticity)')# 4. Residuals histogramsns.histplot(resid, kde=True, bins=40, ax=axes[1,1], color='darkorange')axes[1,1].set_title('Residual Distribution\n(bell shape = normal)')plt.suptitle('OLS Regression Diagnostic Plots', fontsize=13, y=1.01)plt.tight_layout()plt.show()# Durbin-Watsonfrom statsmodels.stats.stattools import durbin_watsondw = durbin_watson(resid)print(f"Durbin-Watson statistic: {dw:.3f} (target: ~2.0 for independence)")print(f"Interpretation: {'OK ' if 1.5 < dw < 2.5 else 'Possible autocorrelation '}")

---
## Task 7 — Chi-Square Tests

### Theory: Testing Associations Between Categorical Variables

When both variables are categorical, we cannot use t-tests or ANOVA (which require numeric outcomes). The Chi-square family of tests handles categorical-categorical associations.

### Chi-Square Test of Independence

Tests whether two categorical variables are statistically independent — i.e., whether the distribution of one variable is the same across all categories of the other.

**H0:** Variable A and Variable B are independent (no association)
**H1:** Variable A and Variable B are associated

The test compares observed frequencies (the actual counts in each cell of a cross-tabulation) against expected frequencies (what we would expect if H0 were true):

```
Expected = (row_total * column_total) / grand_total
Chi-square = sum((Observed - Expected)^2 / Expected)
```

Under H0, the Chi-square statistic follows a chi-squared distribution with (r-1)(c-1) degrees of freedom, where r = number of rows and c = number of columns.

**Assumption:** Expected frequency in each cell >= 5. If this is violated, use Fisher's Exact Test.

### Cramer's V — Effect Size for Chi-Square

The Chi-square statistic depends on both the strength of the association AND the sample size. Cramer's V standardizes it:

```
V = sqrt(Chi-square / (n * (min(r,c) - 1)))
```

| V | Interpretation |
|---|---|
| 0.0 – 0.1 | Negligible association |
| 0.1 – 0.3 | Small to moderate association |
| 0.3 – 0.5 | Moderate to strong association |
| > 0.5 | Strong association |

Always report Cramer's V alongside the Chi-square test result — the test p-value tells you if the association exists; V tells you how strong it is.

### Chi-Square Goodness-of-Fit

A different application: tests whether an observed frequency distribution matches a hypothesized (expected) distribution.

Example: Are orders uniformly distributed across the 7 days of the week? H0 says each day should have 1/7 of all orders. The goodness-of-fit test tells you whether the observed day-of-week distribution differs significantly from uniform.

```
Chi-square = sum((Observed_i - Expected_i)^2 / Expected_i)
df = k - 1  (where k = number of categories)
```

### When to Use Fisher's Exact Test

Fisher's Exact Test should replace Chi-square when:
- The contingency table is 2x2, AND
- Any expected cell frequency is less than 5

Fisher's test computes the exact probability of observing the data, rather than approximating with a chi-squared distribution. It is always valid for 2x2 tables but is computationally intensive for larger tables.


In [ ]:
def chi2_report(var1, var2, data):"""Full chi-square independence test with Cramér's V."""ct = pd.crosstab(data[var1], data[var2])chi2, p, dof, expected = sp.chi2_contingency(ct)n = ct.sum().sum()min_dim = min(ct.shape[0]-1, ct.shape[1]-1)cramers_v = np.sqrt(chi2 / (n * min_dim))effect = 'weak' if cramers_v<0.1 else ('moderate' if cramers_v<0.3 else 'strong')print(f"\n{'='*55}")print(f" Chi-Square: {var1} vs {var2}")print(f"{'='*55}")print(f" H₀: {var1} is independent of {var2}")print(f" χ² = {chi2:.4f} df = {dof} p = {p:.6f}")print(f" Decision: {'REJECT H₀ — significant association' if p<0.05 else 'Fail to Reject H₀'}")print(f" Cramér's V = {cramers_v:.4f} → {effect} association")return ctct1 = chi2_report('return_flag', 'product_category', df)ct2 = chi2_report('payment_method', 'region', df)ct3 = chi2_report('gender', 'product_category', df)

In [ ]:
# Visualise: return rate by product_categoryret_by_cat = df.groupby('product_category')['return_flag'].mean() * 100ret_by_cat = ret_by_cat.sort_values(ascending=False)fig, ax = plt.subplots(figsize=(10, 5))colors = ['#e74c3c' if v > ret_by_cat.mean() else 'steelblue' for v in ret_by_cat]ret_by_cat.plot(kind='bar', ax=ax, color=colors)ax.axhline(ret_by_cat.mean(), color='black', linestyle='--',label=f'Avg {ret_by_cat.mean():.1f}%')ax.set_title('Return Rate % by Product Category\n(red = above average)')ax.set_ylabel('Return Rate %')ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')ax.legend()for i, v in enumerate(ret_by_cat):ax.text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=9)plt.tight_layout()plt.show()

---
## Assignment 3 — Statistical Analysis — Complete! Well done! Proceed to the next assignment notebook. | Assignment | Notebook File |
|---|---|
| 1. Data Cleaning | `A1_Data_Cleaning.ipynb` |
| 2. EDA | `A2_EDA.ipynb` |
| 3. Statistical Analysis | `A3_Statistical_Analysis.ipynb` |
| 4. Segmentation & Clustering | `A4_Segmentation_Clustering.ipynb` |
| 5. Time Series Forecasting | `A5_Time_Series_Forecasting.ipynb` |